In [ ]:
import xarray as xr
import dask
import xclim
from xclim.core.calendar import percentile_doy
from xclim.indices import warm_spell_duration_index
from xclim.indices import tx90p
dask.config.set(**{'array.slicing.split_large_chunks': False})
import numpy as np
from numba import jit
import numba
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
sns.set_palette("colorblind")
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'
rcParams['font.weight'] = 'light'
rcParams['mathtext.fontset'] = 'cm'
rcParams['mathtext.rm'] = 'serif'
mpl.rcParams["figure.dpi"] = 500
import cartopy.crs as ccrs
import cartopy as ct
import matplotlib.colors as c
import random
import regionmask
import cmasher as cmr

regionmask.__version__

In [ ]:
# Opening TREFHTMN standard 90% thresholds
std_thresh = xr.open_dataset('/glade/work/ivyglade/extremes/90_per_thresholds/trefhtmx_90per_thresholds_all_other_cases.nc')['__xarray_dataarray_variable__'].values

In [ ]:
# TREFHTMN data
cont_trefhtmx_6 = xr.open_mfdataset('/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.006/atm/proc/tseries/day_1/' + \
                                       'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.006.cam.h1.TREFHTMX.*.nc', combine='nested', concat_dim='time')

cont_trefhtmx_7 = xr.open_mfdataset('/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007/atm/proc/tseries/day_1/' + \
                                       'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h1.TREFHTMX.*.nc', combine='nested', concat_dim='time')

cont_trefhtmx_8 = xr.open_mfdataset('/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.008/atm/proc/tseries/day_1/' + \
                                       'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.008.cam.h1.TREFHTMX.*.nc', combine='nested', concat_dim='time')

cont_trefhtmx_9 = xr.open_mfdataset('/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.009/atm/proc/tseries/day_1/' + \
                                       'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.009.cam.h1.TREFHTMX.*.nc', combine='nested', concat_dim='time')

cont_trefhtmx_10 = xr.open_mfdataset('/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.010/atm/proc/tseries/day_1/' + \
                                       'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.010.cam.h1.TREFHTMX.*.nc', combine='nested', concat_dim='time')

In [ ]:
cont_trefhtmx = xr.concat([cont_trefhtmx_6, cont_trefhtmx_7, cont_trefhtmx_8, cont_trefhtmx_9, cont_trefhtmx_10], \
                          dim='ens')

In [ ]:
cont_trefhtmx_1568 = cont_trefhtmx.isel(time=slice(0, 365*55-31))['TREFHTMX'].transpose('ens', 'lat', 'lon', 'time')

In [ ]:
# Step two: need to reshape into dimensions of (time, lat, lon) and repeat time 20 times, so we have 20 years of data
trefhtmx_90_thresh_re = std_thresh.swapaxes(0, 2).swapaxes(1, 2)

trefhtmx_90_thresh_20yr = np.zeros((20075, 192, 288))
for i in range(55):
    trefhtmx_90_thresh_20yr[i*365:i*365+365] = trefhtmx_90_thresh_re

trefhtmx_90_thresh_20yr_nodec = trefhtmx_90_thresh_20yr[0:20075-31]
# To put the 95% threshold in the final form for running the exceedance function, we need to swapaxes one more time
trefhtmx_thresh_90 = trefhtmx_90_thresh_20yr_nodec.swapaxes(0, 2).swapaxes(0, 1)

In [ ]:
np.shape(trefhtmx_thresh_90)

In [ ]:
@numba.jit(nopython=True, parallel=True)
def exceedance(per90, data, time):
    exceedances = np.zeros((192, 288, time))
    for i in range(192):
        for j in range(288):
            for l in range(time):
                if data[i][j][l] > per90[i][j][l]:
                    exceedances[i][j][l] = 1
                else:
                    continue
    return exceedances

In [ ]:
@numba.jit(nopython=True, parallel=True)
def heatwaves(exceedances, years):
    total_waves = np.zeros((192, 288, years))
    for i in range(192):
        for j in range(288):
            for l in range(years):
                if l < 54:
                    year_of_int = exceedances[i][j][l*365:l*365+365]
                    count = 0
                    heatwave = 0
                    for ll in range(365):
                        if year_of_int[ll] == 1:
                            count += 1
                            if count == 6:
                                heatwave += 1
                        else: 
                            count = 0
                elif l == 54:
                    year_of_int = exceedances[i][j][l*365:l*365+(365-31)]
                    count = 0
                    heatwave = 0
                    for ll in range(334):
                        if year_of_int[ll] == 1:
                            count += 1
                            if count == 6:
                                heatwave += 1
                        else: 
                            count = 0
                total_waves[i][j][l] = heatwave 
    return total_waves

In [ ]:
# Calculating exceedances for each time series
std_ex = np.zeros((5, 192, 288, 20044))
for i in range(5):
    std_ex[i] = exceedance(trefhtmx_thresh_90, cont_trefhtmx_1568[i].values, 20044)
    # cont_trefhtmn_2039_waves_90[i] = heatwaves(ex_90, 54)

In [ ]:
std_waves = np.zeros((5, 192, 288, 55))
for i in range(5):
    std_waves[i] = heatwaves(std_ex[i], 55)

In [ ]:
test = std_waves.swapaxes(0, 3)[45:55].mean(axis=(0, 3))

In [ ]:
fig, ax = plt.subplots(subplot_kw=dict(projection=ccrs.Robinson()))

norm = c.BoundaryNorm(np.arange(0, 5.5, 0.5), plt.get_cmap('YlOrBr').N)

ax.add_feature(ct.feature.OCEAN, edgecolor='xkcd:gunmetal', facecolor='xkcd:light gray', lw=0.25, zorder=100)
ax.pcolormesh(lon, lat, test, transform=ccrs.PlateCarree(), norm=norm, cmap='YlOrBr')

In [ ]:
# dimensions
lat = cont_trefhtmx_1568['lat']
lon = cont_trefhtmx_1568['lon']
year = np.arange(2015, 2070, 1)
ens = np.arange(1, 6, 1)
time = cont_trefhtmx_1568['time']

In [ ]:
# convert to XR
std_ex_xr = xr.DataArray(std_ex, coords={'lat':lat, 'lon':lon, 'time':time, 'ens':ens}, \
                         dims=['ens', 'lat', 'lon', 'time'])

std_waves_xr = xr.DataArray(std_waves, coords={'lat':lat, 'lon':lon, 'year':year, 'ens':ens}, \
                            dims=['ens', 'lat', 'lon', 'year'])

In [ ]:
# save as a *.nc
std_ex_xr.to_netcdf('/glade/work/ivyglade/extremes/exceedances/cont_ex_1569_all_other_cases_trefhtmx_seasonal.nc')
# std_waves_xr.to_netcdf('/glade/work/ivyglade/extremes/occurrences/cont_waves_1569_all_other_cases_trefhtmx.nc')